# Timeseries Forecasting — Starter Kit

A hands-on tour of using **timeseries foundation models (TFMs)** on small bundled datasets.
Two models are supported:

| Model | Strengths | Limitations |
|---|---|---|
| **Chronos-2** (Amazon, `amazon/chronos-2`) | Multivariate, supports covariates | Larger |
| **TimesFM 2.5** (Google, `google/timesfm-2.5-200m-pytorch`) | Compact, fast, long context | Univariate only, no covariates |

Each step is a single code cell with a few configurable variables at the top. Read the comments,
adjust the variables, and re-run the cell.

> **Setup**: see `getting-started.md` for installing the libraries and selecting a virtual environment.


In [ ]:
# --- Standard imports ---
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Quiet noisy deprecation warnings from the model libraries.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Expected ./data folder next to this notebook."


## 1. Load a dataset

The `./data` folder ships with several bundled CSVs. Pick one by filename below.
All bundled files use European date format (day-first), e.g. `01/01/2024 00:00`.

The cell prints the head of the dataframe and plots the whole dataset for a quick visual check.


In [ ]:
# Pick a bundled dataset. Available files in ./data:
#
#   "Total-load-on-Belgian-grid.csv"       —  15-min  —  target: "Total Load MW"
#   "Belgian-offshore-wind-production.csv" —  15-min  —  target: "Total Production MW"
#   "DAM-prices-BELPEX.csv"                —  1-hour  —  target: "Eur/MWh"
#   "Oil-Temperature-ETTh1.csv"            —  1-hour  —  target: "Oil Temperature"
#   "Seoul-Bike-Demand.csv"                —  1-hour  —  target: "Rented Bike Count"

DATASET = "Total-load-on-Belgian-grid.csv"

# ------------------------------------------------------------
path = DATA_DIR / DATASET
assert path.exists(), f"Dataset not found: {path}"

df = pd.read_csv(path, encoding="utf-8-sig")

# First column is the timestamp (true for all bundled files).
ts_col = df.columns[0]
df[ts_col] = pd.to_datetime(df[ts_col], dayfirst=True, errors="raise")
df = df.set_index(ts_col).sort_index()

print(f"Loaded {DATASET}: {len(df):,} rows × {len(df.columns)} columns")
print(f"Date range : {df.index[0]} → {df.index[-1]}")
print(f"Columns    : {list(df.columns)}")
print()
print("Head:")
display(df.head())

# Plot every numeric column
fig, ax = plt.subplots(figsize=(12, 4))
df.select_dtypes(include="number").plot(ax=ax, lw=0.7)
ax.set_title(f"{DATASET} — full dataset")
ax.set_ylabel("value")
ax.set_xlabel("")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Choose a model

Set `MODEL` to one of two values. Each model requires a separate `pip install` —
see `getting-started.md`.


In [ ]:
# "chronos-2"   → amazon/chronos-2                  (multivariate, supports covariates)
# "timesfm-2.5" → google/timesfm-2.5-200m-pytorch   (univariate, no covariates)

MODEL = "chronos-2"

assert MODEL in {"chronos-2", "timesfm-2.5"}, f"Unknown MODEL: {MODEL!r}"
print(f"Will use: {MODEL}")


## 3. Set forecasting parameters

These variables control *what* gets predicted. Defaults assume the
Belgian load dataset (15-minute granularity). For other datasets, adjust to taste —
remember `HISTORY` and `HORIZON` are counted in **steps**, not hours.

Covariates apply only when `MODEL == "chronos-2"`.


In [ ]:
# Column to forecast. For multivariate datasets, just pick one column.
TARGET = "Total Load MW"

# Number of past steps fed to the model as context.
HISTORY = 672      # 168 hours = 1 week at 15-min granularity

# Number of future steps to predict.
HORIZON = 96       # 24 hours at 15-min granularity

# When to start forecasting.
#   - None                  → use the last HORIZON rows for backtest (actuals available)
#   - "YYYY-MM-DD HH:MM"    → explicit timestamp inside the data range
FORECAST_START = None

# Covariates — chronos-2 only. Comment out lines to disable individual features;
# set COVARIATES = [] for no covariates at all.
COVARIATES = [
    "Holidays (BE)",
    "Day of week",
    "Hour of day",
]

# ------------------------------------------------------------
assert TARGET in df.columns, f"TARGET '{TARGET}' not in dataset columns: {list(df.columns)}"

if FORECAST_START is None:
    fc_start = df.index[-HORIZON]
else:
    fc_start = pd.to_datetime(FORECAST_START)
    assert fc_start in df.index, (
        f"FORECAST_START {fc_start} is not in the dataset index.\n"
        f"Pick a timestamp between {df.index[0]} and {df.index[-1]}."
    )

fc_pos = df.index.get_loc(fc_start)
history_start_pos = max(0, fc_pos - HISTORY)

history_df = df.iloc[history_start_pos:fc_pos]
future_index = df.index[fc_pos:fc_pos + HORIZON]

assert len(history_df) > 0, "History window is empty — reduce HISTORY or move FORECAST_START later."
assert len(future_index) > 0, "Future window is empty — move FORECAST_START earlier."

is_backtest = len(future_index) == HORIZON

print(f"Target          : {TARGET}")
print(f"History window  : {history_df.index[0]} → {history_df.index[-1]}  ({len(history_df)} steps)")
print(f"Forecast window : {future_index[0]} → {future_index[-1]}  ({len(future_index)} steps)")
print(f"Backtest        : {'yes (actuals available)' if is_backtest else 'no (forward forecast)'}")
print(f"Covariates      : {COVARIATES if MODEL == 'chronos-2' else '— (not supported by this model)'}")


## 4. Generate the forecast

Each model has its own API. The code below picks the right one based on the `MODEL`
variable. Calendar covariates (`Holidays (BE)` / `Day of week` / `Hour of day`) are
computed from timestamps — no extra data files needed.

The first time you run a model, its weights are downloaded from HuggingFace
(roughly 0.5–4 GB depending on the model) into `~/.cache/huggingface/`. Subsequent
runs reuse the cache.


In [ ]:
import holidays


def build_covariates(idx: pd.DatetimeIndex, selected: list[str]) -> pd.DataFrame:
    """Build a small DataFrame of calendar covariates for the given timestamps.

    Only the three calendar features documented in section 3 are supported.
    """
    out = pd.DataFrame(index=idx)
    if "Day of week" in selected:
        out["dow"] = idx.dayofweek.values.astype(float)
    if "Hour of day" in selected:
        out["hour"] = idx.hour.values.astype(float)
    if "Holidays (BE)" in selected:
        years = sorted({int(d.year) for d in idx})
        be_cal = holidays.country_holidays("BE", years=years)
        out["holiday"] = np.array(
            [1.0 if d.date() in be_cal else 0.0 for d in idx], dtype=float
        )
    return out


median = lower = upper = None

if MODEL == "chronos-2":
    # --- Chronos-2 -----------------------------------------------------------
    # API: pipeline.predict_df(context_df, future_df, ...)
    #   context_df : timestamp + item_id + target column + (optional) covariates
    #   future_df  : timestamp + item_id + future covariates (no target)
    from chronos import BaseChronosPipeline
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32

    print(f"Loading amazon/chronos-2 on {device} ...")
    pipeline = BaseChronosPipeline.from_pretrained(
        "amazon/chronos-2", device_map=device, torch_dtype=dtype,
    )

    ctx_cov = build_covariates(history_df.index, COVARIATES)
    fut_cov = build_covariates(future_index, COVARIATES)

    context_df = pd.DataFrame({
        "timestamp": history_df.index,
        "item_id": "0",
        TARGET: history_df[TARGET].astype(float).values,
        **{c: ctx_cov[c].values for c in ctx_cov.columns},
    })
    future_df = None
    if len(fut_cov.columns) > 0:
        future_df = pd.DataFrame({
            "timestamp": future_index,
            "item_id": "0",
            **{c: fut_cov[c].values for c in fut_cov.columns},
        })

    pred_df = pipeline.predict_df(
        context_df,
        future_df=future_df,
        id_column="item_id",
        timestamp_column="timestamp",
        target=TARGET,
        prediction_length=HORIZON,
        quantile_levels=[0.1, 0.5, 0.9],
    )

    # Returned DataFrame has quantile columns named either as floats (0.5)
    # or strings ("0.5"). Tolerate both.
    def _col(level: float):
        for key in (level, str(level), f"{level:.1f}"):
            if key in pred_df.columns:
                return pred_df[key].to_numpy()
        return None

    median = _col(0.5)
    lower  = _col(0.1)
    upper  = _col(0.9)
    if median is None and "predictions" in pred_df.columns:
        median = pred_df["predictions"].to_numpy()

elif MODEL == "timesfm-2.5":
    # --- TimesFM 2.5 ---------------------------------------------------------
    # API: model.forecast(horizon, inputs=[1-D array])
    # TimesFM is univariate — we feed only the target.
    import timesfm
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # The public from_pretrained currently forwards unknown kwargs (e.g. proxies)
    # to __init__, which trips on newer huggingface_hub. Calling the internal
    # _from_pretrained directly works around this.
    print(f"Loading google/timesfm-2.5-200m-pytorch on {device} ...")
    model = timesfm.TimesFM_2p5_200M_torch._from_pretrained(
        model_id="google/timesfm-2.5-200m-pytorch",
        revision=None,
        cache_dir=None,
        force_download=False,
        local_files_only=False,
        token=None,
    )
    try:
        model = model.to(device)
    except Exception:
        pass

    # TimesFM 2.5 caps max_context + max_horizon at 16,384.
    TOTAL_BUDGET = 16384
    max_context = min(HISTORY, TOTAL_BUDGET - HORIZON)
    model.compile(timesfm.ForecastConfig(
        max_context=max_context,
        max_horizon=HORIZON,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    ))

    ctx = history_df[TARGET].astype(float).to_numpy()[-max_context:]
    point_forecast, quantile_forecast = model.forecast(horizon=HORIZON, inputs=[ctx])

    median = np.asarray(point_forecast[0][:HORIZON], dtype=float)
    qf = np.asarray(quantile_forecast)
    # quantile_forecast shape: (n_series, horizon, 10) — [mean, p10..p90]
    if qf.ndim == 3 and qf.shape[-1] >= 10:
        lower = qf[0, :HORIZON, 1]   # 10th percentile
        upper = qf[0, :HORIZON, 9]   # 90th percentile

# --- Assemble a tidy result DataFrame -------------------------------------
result = pd.DataFrame({
    "timestamp": future_index,
    "forecast": np.asarray(median, dtype=float),
})
if lower is not None and upper is not None:
    result["lower_p10"] = np.asarray(lower, dtype=float)
    result["upper_p90"] = np.asarray(upper, dtype=float)
if is_backtest:
    result["actual"] = df[TARGET].reindex(future_index).astype(float).values
result = result.set_index("timestamp")

print(f"\nDone — forecast generated with {MODEL}.")
print(f"Result columns: {list(result.columns)}")
print()
print("Head of the forecast result:")
display(result.head())


## 5. Visualize and evaluate

Plot the actual series with the forecast overlaid (dashed), and report **WAPE**
(Weighted Absolute Percentage Error) when actuals are available.

$$\text{WAPE} = \frac{\sum_t |y_t - \hat{y}_t|}{\sum_t |y_t|} \times 100\%$$

WAPE is robust to zeros (unlike MAPE) and comparable across series of different scales.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

# Actual line — history + (if backtest) the actual values during the forecast window
actual_x = list(history_df.index)
actual_y = list(history_df[TARGET].astype(float).values)
if is_backtest:
    actual_x.extend(result.index)
    actual_y.extend(result["actual"].values)
ax.plot(actual_x, actual_y, color="#627BF4", lw=1.0,
        label=f"{TARGET} (actual)")

# Forecast line — dashed, contrasting color
ax.plot(result.index, result["forecast"],
        color="#FF715B", linestyle="--", lw=1.5,
        label=f"{TARGET} (forecast)")

# Uncertainty band
if "lower_p10" in result.columns and "upper_p90" in result.columns:
    ax.fill_between(result.index, result["lower_p10"], result["upper_p90"],
                    color="#FF715B", alpha=0.15, label="p10–p90 band")

# Vertical line at forecast start
ax.axvline(result.index[0], color="#888", linestyle=":", lw=1)

ax.set_title(f"Actual vs forecast — {TARGET}   ({MODEL})")
ax.set_xlabel("")
ax.set_ylabel(TARGET)
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- WAPE ----------------------------------------------------------------
if is_backtest:
    actual = result["actual"].to_numpy()
    forecast = result["forecast"].to_numpy()
    denom = float(np.nansum(np.abs(actual)))
    if denom == 0:
        wape = float("nan")
    else:
        wape = float(np.nansum(np.abs(actual - forecast)) / denom * 100.0)
    print(f"WAPE : {wape:.2f}%")
else:
    print("WAPE : — (forward forecast; no actuals to compare against)")


---

That's the loop. To explore further, edit a variable in any step and re-run that step and the ones below it.

Some natural follow-ups:

- **Try the other model** — set `MODEL = "timesfm-2.5"` in step 2 and re-run from step 4 onward.
- **Try a different dataset** — change `DATASET` and `TARGET` in steps 1 and 3.
- **Look at a different period** — set an explicit `FORECAST_START` in step 3.
- **Toggle covariates** — comment lines in/out of `COVARIATES` in step 3 (only matters for Chronos-2).
- **Longer / shorter context** — change `HISTORY`.
